<br/>
<img src="images/utfsm.png" alt="" width="130px" align="left"/>
<img src="images/utfsm.png" alt="" width="130px" align="right"/>
<div align="center">
<h1>Redes Generativas</h1>
<br/><br/>
Dr. Nicolás Gálvez Ramírez<br/>
Dr. Patricio Olivares Roncagliolo<br/><br/>
Ingeniería Civil Telemática<br/>
Departamento de Eléctronica<br/>
Universidad Técnica Federico Santa María
</div>
<br>
Fuentes: 
<br>
"Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow"

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** el entrenamiento adversarial: generador frente a discriminador.
2. **Implementar** una GAN simple y reconocer la arquitectura de una DCGAN.
3. **Entender** los Autoencoders (AE) y los Variational Autoencoders (VAE): cuello de botella, reparametrización y ELBO.
4. **Generar** e interpolar muestras sintéticas en el espacio latente.
5. **Reconocer** problemas de entrenamiento (*mode collapse*) y métricas de evaluación (FID, IS).

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb).

# Auto Encoders (AE)

- Redes Neuronales utilizadas para la reducción de dimensionalidad y la extracción de características.
- Consta de dos partes: un **codificador** (coder) y un **decodificador** (decoder).
- El objetivo de un autoencoder es **minimizar la diferencia entre la entrada ya la reconstrucción**

<img src="images/AE.png" alt="" width="800px" align="center"/>
Fuente: "Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow"

<img src="images/AE2.png" alt="" width="800px" align="center"/>
Fuente: "https://towardsdatascience.com/applied-deep-learning-part-3-autoencoders-1c083af4d798"

## Aplicaciones de AutoEncoders
- Reducción de dimensionalidad (compresión)
- Eliminación de ruido en imágenes
- Detección de anomalías

<img src="images/AE_noise.png" alt="" width="800px" align="center"/>


In [ ]:
import tensorflow as tf
# Ejemplo usando fashion mnist
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Cost",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"] # 10 clases

X_train_full = X_train_full/255.0 # Escalamiento

from sklearn.model_selection import train_test_split
# Utilización de datos de validación y entrenamiento del modelo
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, 
                                                  stratify=y_train_full)

stacked_encoder = tf.keras.models.Sequential([
tf.keras.layers.Flatten(input_shape=[28, 28]),
tf.keras.layers.Dense(100, activation="selu"),
tf.keras.layers.Dense(30, activation="selu"),
])

stacked_decoder = tf.keras.models.Sequential([
tf.keras.layers.Dense(100, activation="selu", input_shape=[30]),
tf.keras.layers.Dense(28 * 28, activation="sigmoid"),
tf.keras.layers.Reshape([28, 28])
])

stacked_ae = tf.keras.models.Sequential([stacked_encoder, stacked_decoder])

stacked_ae.compile(loss="binary_crossentropy",
optimizer=tf.keras.optimizers.SGD(lr=1.5))
history = stacked_ae.fit(X_train, X_train, epochs=50, validation_data=[X_val, X_val])

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(15,15))
prediction = stacked_ae.predict(X_train)

i=19

ax1 = fig.add_subplot(1,2,1)
ax1.imshow(X_train[i], cmap='gist_yarg')
ax1.set_title(class_names[y_train[i]])
ax2 = fig.add_subplot(1,2,2)
ax2.imshow(prediction[i], cmap='gist_yarg')
ax2.set_title(class_names[y_train[i]])


plt.show()

# Convolutional Auto Encoder

- Autoencoders diseñados para el procesamiento de imágenes.
- Utilizan capas convolucionales.
- Las capas convolucionales permiten capturar patrones en las imágenes de manera más eficiente.

In [ ]:
conv_encoder = tf.keras.models.Sequential([
tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28]),
tf.keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="selu"),
tf.keras.layers.MaxPool2D(pool_size=2),
tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="selu"),
tf.keras.layers.MaxPool2D(pool_size=2),
tf.keras.layers.Conv2D(64, kernel_size=3, padding="same", activation="selu"),
tf.keras.layers.MaxPool2D(pool_size=2)
])
conv_decoder = tf.keras.models.Sequential([
tf.keras.layers.Conv2DTranspose(32, kernel_size=3, strides=2, padding="valid",activation="selu", input_shape=[3, 3, 64]),
tf.keras.layers.Conv2DTranspose(16, kernel_size=3, strides=2, padding="same",
activation="selu"),
tf.keras.layers.Conv2DTranspose(1, kernel_size=3, strides=2, padding="same",
activation="sigmoid"),
tf.keras.layers.Reshape([28, 28])
])

conv_ae = tf.keras.models.Sequential([conv_encoder, conv_decoder])

conv_ae.compile(loss="binary_crossentropy",
optimizer=tf.keras.optimizers.SGD(lr=1.5))
history = conv_ae.fit(X_train, X_train, epochs=50, validation_data=[X_val, X_val])

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(15,15))
prediction = conv_ae.predict(X_train)

i=7

ax1 = fig.add_subplot(1,2,1)
ax1.imshow(X_train[i], cmap='gist_yarg')
ax1.set_title(class_names[y_train[i]])
ax2 = fig.add_subplot(1,2,2)
ax2.imshow(prediction[i], cmap='gist_yarg')
ax2.set_title(class_names[y_train[i]])

plt.show()

# Variational Autoencoders (VAE)

- Un Variational Autoencoder (VAE) es una variante de los autoencoders que combina técnicas de reducción de dimensionalidad y generación de datos con modelos generativos probabilísticos
- Un autoencoder tradicional tiene:
    * Un codificador que reduce la dimensionalidad de los datos de entrada a una representación latente
    * Un decodificador que reconstruye los datos originales a partir de la representación latente.
- En un VAE, también tenemos un codificador y un decodificador, pero se introduce un componente probabilístico en el proceso.

<img src="images/VAE_Basic.png" alt="" width="800px" align="center"/>
Fuente: "https://en.wikipedia.org/wiki/Variational_autoencoder"

<img src="images/VAE2.png" alt="" width="800px" align="center"/>
Fuente: "https://towardsdatascience.com/intuitively-understanding-variational-autoencoders-1bfe67eb5daf"

In [ ]:
import tensorflow as tf
class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        return tf.keras.backend.random_normal(tf.shape(log_var)) * tf.keras.backend.exp(log_var / 2) + mean

codings_size = 10
inputs = tf.keras.layers.Input(shape=[28, 28])
z = tf.keras.layers.Flatten()(inputs)
z = tf.keras.layers.Dense(150, activation="selu")(z)
z = tf.keras.layers.Dense(100, activation="selu")(z)
codings_mean = tf.keras.layers.Dense(codings_size)(z) # μ
codings_log_var = tf.keras.layers.Dense(codings_size)(z) # γ
codings = Sampling()([codings_mean, codings_log_var])
variational_encoder = tf.keras.Model(
inputs=[inputs], outputs=[codings_mean, codings_log_var, codings])

decoder_inputs = tf.keras.layers.Input(shape=[codings_size])
x = tf.keras.layers.Dense(100, activation="selu")(decoder_inputs)
x = tf.keras.layers.Dense(150, activation="selu")(x)
x = tf.keras.layers.Dense(28 * 28, activation="sigmoid")(x)
outputs = tf.keras.layers.Reshape([28, 28])(x)
variational_decoder = tf.keras.Model(inputs=[decoder_inputs], outputs=[outputs])

In [ ]:
_, _, codings = variational_encoder(inputs)
reconstructions = variational_decoder(codings)
variational_ae = tf.keras.Model(inputs=[inputs], outputs=[reconstructions])

latent_loss = -0.5 * tf.keras.backend.sum(
1 + codings_log_var - tf.keras.backend.exp(codings_log_var) - tf.keras.backend.square(codings_mean),
axis=-1)
variational_ae.add_loss(tf.keras.backend.mean(latent_loss) / 784.)
variational_ae.compile(loss="binary_crossentropy", optimizer="rmsprop")

In [ ]:
history = variational_ae.fit(X_train, X_train, epochs=50, batch_size=128,
validation_data=[X_val, X_val])

In [ ]:
codings = tf.random.normal(shape=[12, codings_size])
images = variational_decoder(codings).numpy()

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(3,3))
prediction = conv_ae.predict(X_train)

i=8

ax1 = fig.add_subplot(1,2,1)
ax1.imshow(images[i], cmap='gist_yarg')
plt.show()

En los VAE, el codificador crea instancias a partir del muestreo de las variables latentes, lo que habilita a estos modelos para generar datos distintos de los que se utilizaron en su entrenamiento inicial.

# Generative Adversarial Networks (GAN)

Las Redes Generativas Adversarias (GAN por sus siglas en inglés), es una arquitectura de red neuronal que se utiliza para generar datos.
Consta de dos componentes:
- El Generador: El cual crea nuevos datos a partir de un ruido aleatorio o un vector de entrada
- El Discriminador: Este evalúa si un dato es real (proveniente de un conjunto de entrenamiento real) o falso (generado por el generador)

<img src="images/GAN.png" alt="" width="800px" align="center"/>
Fuente: "https://www.microsoft.com/en-us/research/blog/how-can-generative-adversarial-networks-learn-real-life-distributions-easily/"

## Entrenamiento de una red GAN

El entrenamiento consta de dos etapas:
- **Entrenamiento del discriminador**: En la primera fase, alimentamos al generador con ruido gaussiano para producir imágenes falsas. Completamos este lote concatenando un número igual de imágenes reales. Las etiquetas se establecen en 0 para imágenes falsas y 1 para imágenes reales. Luego, entrenamos al discriminador en este lote.
- **Entrenamiento del generador**: En la segunda fase, suministramos ruido gaussiano al modelo GAN. El generador producirá inicialmente imágenes falsas y el discriminador intentará determinar si estas imágenes son falsas o reales. Queremos que el discriminador crea que las imágenes falsas son reales, por lo que las etiquetas se establecen en 1.

In [ ]:
import tensorflow as tf
codings_size = 30
generator = tf.keras.models.Sequential([
tf.keras.layers.Dense(100, activation="selu", input_shape=[codings_size]),
tf.keras.layers.Dense(150, activation="selu"),
tf.keras.layers.Dense(28 * 28, activation="sigmoid"),
tf.keras.layers.Reshape([28, 28])
])

discriminator = tf.keras.models.Sequential([
tf.keras.layers.Flatten(input_shape=[28, 28]),
tf.keras.layers.Dense(150, activation="selu"),
tf.keras.layers.Dense(100, activation="selu"),
tf.keras.layers.Dense(1, activation="sigmoid")
])
gan = tf.keras.models.Sequential([generator, discriminator])

discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(1000)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)

In [ ]:
from tqdm import tqdm
def train_gan(gan, dataset, batch_size, codings_size, n_epochs=50):
    generator, discriminator = gan.layers
    for epoch in tqdm(range(n_epochs)):
        for X_batch in dataset:
            # phase 1 - training the discriminator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            generated_images = tf.cast(generator(noise),tf.float32)
            X_batch = tf.cast(X_batch, tf.float32) 
            X_fake_and_real = tf.concat([generated_images, X_batch], axis=0)
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
            discriminator.trainable = True # Acá el discriminador se marca como entrenable
            discriminator.train_on_batch(X_fake_and_real, y1)
            # phase 2 - training the generator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            y2 = tf.constant([[1.]] * batch_size)
            discriminator.trainable = False
            gan.train_on_batch(noise, y2)
train_gan(gan, dataset, batch_size, codings_size)

In [ ]:
noise = tf.random.normal(shape=[1, codings_size])
image = generator(noise).numpy().reshape((28,28))

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(3,2))

ax1 = fig.add_subplot(1,1,1)
ax1.imshow(image, cmap='gist_yarg')
plt.show()

In [ ]:
from IPython.core.display import HTML
HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")
#codigo extra, para que imagenes de matplotlib
#estén centradas en las diapositivas, ejecutar antes de lanzar los ejemplos.